### Get Imports

In [6]:
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

MODEL = 'gpt-4.1-mini'
openai = OpenAI()


### Check if API key is accessible 

In [10]:
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    

API key looks good so far


### System and User Prompt for getting relevant links from the website

In [12]:
link_system_prompt ="""
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about a company,
such as links to an About page, or a Company page, or Careers/Jobs page.
You should respond in JSON as in this example:

{
    "links" : [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company,
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

print(get_links_user_prompt("https://huggingface.co"))


Here is the list of links on the website https://huggingface.co -
Please decide which of these are relevant web links for a brochure about the company,
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

/
/models
/datasets
/spaces
/storage
/docs
/enterprise
/pricing
/tasks
/chat
/collections
/languages
/organizations
/blog
/posts
/papers
/hardware
/learn
/join/discord
https://discuss.huggingface.co/
https://github.com/huggingface
/enterprise
/pro
/support
/inference/models
/inference-endpoints
/storage
/login
/join
/spaces
/models
/Qwen/Qwen3.8-27B
/unsloth/Qwen3.8-27B-GGUF
/Qwen/Qwen3.8-2.4T-A95B
/Lightricks/LTX-2.5
/MiniMaxAI/MiniMax-Music3
/models
/spaces/MiniMaxAI/MiniMax-Music3
/spaces/MiniMaxAI/MiniMax-H3-Turbo-Lora
/spaces/prithivMLmods/Qwen-Image-Edit-2511-LoRAs-Fast
/spaces/agent-memory-leaderboard/leaderboard
/spaces/thornmaze/reel-lab
/spaces
/datasets/r0b0tlab/qwen3.8-max-glm5.2-kim

### Function to retrive relevant links from the website using the model

In [13]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model = MODEL,
        messages = [
            {"role": "system", "content" : link_system_prompt},
            {"role": "user", "content" : get_links_user_prompt(url)}
        ]
    )
    results = response.choices[0].message.content
    links = json.loads(results)
    return links

select_relevant_links("https://huggingface.co")

{'links': [{'type': 'about page', 'url': 'https://huggingface.co/huggingface'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'blog page', 'url': 'https://huggingface.co/blog'},
  {'type': 'community forum', 'url': 'https://discuss.huggingface.co/'},
  {'type': 'company LinkedIn',
   'url': 'https://www.linkedin.com/company/huggingface/'}]}

### Function to get all information regarding the webpage 

In [14]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

print(fetch_page_and_all_relevant_links("https://huggingface.co"))

## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Website
Tasks
HuggingChat
Collections
Languages
Organizations
Community
Blog
Posts
Daily Papers
Hardware
Learn
Discord
Forum
GitHub
Solutions
Team & Enterprise
Hugging Face PRO
Enterprise Support
Inference Providers
Inference Endpoints
Storage Buckets
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
Qwen/Qwen3.8-27B
Updated
4 days ago
•
666k
•
11k
unsloth/Qwen3.8-27B-GGUF
Updated
3 days ago
•
3.56M
•
1.76k
Qwen/Qwen3.8-2.4T-A95B
Updated
6 days ago
•
11.2k
•
1.06k
Lightricks/LTX-2.5
Updated
1 day ago
•
504k
•
1.18k
MiniMaxAI/MiniMax-Music3
Updated
4 days ago
•
11.7k
•
932
Browse 2M+ models
Spaces
Running
on
Zero
Agents
Featured
159
MiniMax Music 3 Studio
🎵
159
Generate custom 

### System and User prompt for Brochure Generation

In [15]:
brochure_system_prompt ="""
You are a assistant that analyzes the contents of several relevant pages from a company website
and create a short brochure about the company for propective customers, investors and recruits.
Respond in mardown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
Use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""

    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000]
    return user_prompt 

### Fucntion to generate the brochure

In [16]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model= MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream = True 
    )
    response = ""
    display_handle = display(Markdown(""), display_id = True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response),display_id=display_handle.display_id)

stream_brochure("HuggingFace", "https://huggingface.com")

# Hugging Face Brochure

---

## About Hugging Face

**Hugging Face** is a vibrant AI community and platform dedicated to building the future of machine learning (ML). It serves as a collaborative hub where the global ML community creates, shares, and collaborates on thousands of machine learning models, datasets, applications, and tools, fostering innovation and accelerating the adoption of AI.

---

## What We Offer

- **Model Hub:** Access and contribute to over 2 million pre-trained models covering a wide range of AI tasks including natural language processing, computer vision, and multimodal applications.
- **Datasets:** Explore and share from a vast library of more than 500,000 datasets, catering to diverse machine learning needs.
- **Spaces:** Run and host AI-powered applications (over 1 million) using shared resources and zero-cost agents.
- **Buckets & Storage:** Enterprise-grade storage solutions optimized for ML workflows.
- **Enterprise Solutions:** Specialized services including inference endpoints, enterprise support, and enterprise-grade security.
- **HuggingChat:** Cutting-edge AI chat applications and conversational AI tools.
- **Learning & Community:** Active forums, Discord, GitHub repositories, blogs, papers, and educational tracks to help ML practitioners grow.

---

## Company Culture & Community

Hugging Face prides itself on fostering an inclusive, open-source culture that thrives on transparency and collaboration. It is home to over 186 core team members and a flourishing user base exceeding 100,000 AI and ML enthusiasts worldwide. The community contributes actively through forums, shared projects, research papers, and by hosting large-scale datasets and models.

Core values include:
- **Open Collaboration:** Empowering people around the globe to develop and improve machine learning technology.
- **Innovation:** Staying at the forefront of AI research and practical application.
- **Accessibility:** Democratizing AI by making tools and knowledge freely available.

---

## Our Customers & Partners

Hugging Face supports:
- AI researchers and developers building the next generation of models.
- Enterprises integrating AI into their products and services with scalable infrastructure.
- Educational institutions and students learning ML fundamentals.
- Open-source contributors developing tools, datasets, and applications.

Enterprises benefit from Hugging Face PRO and customized support for deploying AI at scale reliably.

---

## Careers at Hugging Face

Hugging Face is expanding and actively hiring passionate, driven individuals in areas including:
- Machine Learning Research and Engineering
- Software Development
- Data Science and Engineering
- Product Management
- Developer Relations and Community Management

The company offers a creative, mission-driven work environment focused on impact, learning, and community engagement. Join a team dedicated to shaping the AI future in a collaborative and open environment.

Current openings and application details are available on the Hugging Face careers page.

---

## Connect & Learn More

- Visit: [huggingface.co](https://huggingface.co)  
- Join the Community on Discord, Forum, and GitHub  
- Explore Tutorials, Papers, and Blog at Hugging Face  
- Follow Hugging Face on Social Media for latest updates  

---

## Brand Assets

Hugging Face provides a comprehensive collection of official brand assets, including logos and color palettes, for use in projects — promoting consistent and vibrant branding aligned with the company’s identity.

---

**Hugging Face**  
*The AI community building the future.*

---